In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nagalakshmi00998y/bpccpart8/bpcc_telugu_fixed100k samples.csv
/kaggle/input/datasets/nagalakshmi00998y/test-set/FLORES200_en_te_devtest_UTF8.csv
/kaggle/input/datasets/nagalakshmi00998y/test-set/IN22_Conv_EN_TE (1).xlsx
/kaggle/input/datasets/nagalakshmi00998y/test-set/IN22_Gen_en_te.csv
/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/__notebook__.ipynb
/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/__output__.json
/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/__results__.html
/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/custom.css
/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/__results___files/__results___4_1.png
/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/__results___files/__results___4_2.png
/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/__results___files/__results___4_3.png
/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/pscab_q

In [2]:
"""
================================================================================
PART 8  -  QLoRA fine-tune of an LLM (Gemma-2-2B-it) for English->Telugu  (reference LLM baseline)
================================================================================
Run in a NEW Kaggle notebook, ONE cell, FRESH session.
  Settings: GPU T4, Internet ON.  + Add Input: your PS-CAB output (8-10queue-pscabpart4).
  Add-ons -> Secrets: HF_TOKEN = your Hugging Face read token (and accept the Gemma licence on HF).
Trains on N_TRAIN pairs from pscab_q1/data/train.csv (same frozen split as your paper), 1 epoch,
then translates: a fixed 1000-sentence sample of test + all 3 external sets, and scores
BLEU / chrF++ / COMET. Your mBART baseline / PS-CAB are scored on the SAME sentences.
Resumable: re-run the cell after a disconnect, it continues from the last checkpoint / saved predictions.
If the log says "LOSS IS NaN", set MODEL = "Qwen/Qwen2.5-1.5B-Instruct" and run again.
================================================================================
"""
MODEL = "unsloth/gemma-2-2b-it"        # same Gemma-2-2B-it weights, public mirror: no HF token needed
N_TRAIN = 8000
N_TEST_SAMPLE = 1000

import os, sys, glob, json, subprocess, importlib.metadata as md
OUT, TMP = "/kaggle/working/part8", "/tmp/part8"
os.makedirs(OUT, exist_ok=True); os.makedirs(TMP, exist_ok=True)

def sh(cmd, check=True):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0 and check:
        print(r.stdout[-2500:]); print(r.stderr[-2500:]); raise RuntimeError("failed: " + " ".join(cmd[:8]))
    return r

try:
    g = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True); HAS_GPU = g.returncode == 0
except FileNotFoundError:
    HAS_GPU = False
if not HAS_GPU:
    raise SystemExit("[STOP] No GPU. Session options -> Accelerator -> GPU T4, then run again.")

cands = []
for d in range(1, 7): cands += glob.glob("/kaggle/input/" + "/".join(["*"] * d) + "/pscab_q1/preds")
if not cands: raise FileNotFoundError("Attach your PS-CAB output (the one with pscab_q1/).")
PRED = max(cands, key=lambda p: len(glob.glob(p + "/*.csv")))
ROOT = os.path.dirname(PRED)
TRAIN = f"{ROOT}/data/train.csv"
if not os.path.exists(TRAIN):
    alt = glob.glob("/kaggle/input/**/pscab_q1/data/train.csv", recursive=True)
    if not alt: raise FileNotFoundError(f"{TRAIN} not found. Attach the output that contains pscab_q1/data/train.csv")
    TRAIN = alt[0]
print("[input] preds:", PRED); print("[input] train:", TRAIN)

HF_TOKEN = os.environ.get("HF_TOKEN", "")
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("[info] no HF_TOKEN secret attached (not needed for unsloth/gemma-2-2b-it or Qwen).")

frozen = []
for p in ["numpy", "scipy", "pandas", "scikit-learn", "protobuf", "torch", "torchvision", "torchaudio", "pillow", "pyarrow"]:
    try: frozen.append(f"{p}=={md.version(p)}")
    except md.PackageNotFoundError: pass
cons = f"{TMP}/c.txt"; open(cons, "w").write("\n".join(frozen) + "\n")
PIP = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-c", cons]
print("[pip] installing (2-4 min) ...")
sh(PIP + ["transformers==4.48.3", "peft==0.14.0", "accelerate==1.3.0", "bitsandbytes", "sentencepiece", "sacrebleu",
          "entmax", "jsonargparse", "pytorch-lightning", "torchmetrics", "huggingface_hub", "pyyaml", "psutil", "packaging"])
sh(PIP + ["--no-deps", "unbabel-comet"])

WORKER = r'''
import os, sys, json, glob, math, time, random
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"; os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np, pandas as pd, torch, sacrebleu
cfg = json.load(open(sys.argv[1])); MODEL, PRED, TRAIN, OUT = cfg["model"], cfg["pred"], cfg["train"], cfg["out"]
N_TRAIN, N_TS = cfg["n_train"], cfg["n_test"]
from huggingface_hub import login
if os.environ.get("HF_TOKEN"): login(os.environ["HF_TOKEN"])
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments, TrainerCallback)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
TAG = MODEL.split("/")[-1]
print("[ver] torch", torch.__version__, "| gpu", torch.cuda.get_device_name(0), flush=True)

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token
END = "<end_of_turn>" if "gemma" in MODEL.lower() else tok.eos_token
def prompt(src):
    return tok.apply_chat_template([{"role": "user", "content":
        "Translate the following English sentence into Telugu. Output only the Telugu translation.\nEnglish: " + src}],
        tokenize=False, add_generation_prompt=True)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.float16)
def base():
    return AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, torch_dtype=torch.float16,
                                                attn_implementation="eager", device_map={"": 0})
ADAPTER = f"{OUT}/adapter_{TAG}"

# ---------------- training ----------------
if not os.path.exists(f"{ADAPTER}/adapter_config.json"):
    tr = pd.read_csv(TRAIN, dtype=str, keep_default_na=False)
    tr = tr[(tr["en"].str.len() > 0) & (tr["te"].str.len() > 0)].sample(n=min(N_TRAIN, len(tr)), random_state=42)
    feats = []
    for s, t in zip(tr["en"], tr["te"]):
        p = tok(prompt(s), add_special_tokens=False)["input_ids"]; y = tok(t + END, add_special_tokens=False)["input_ids"]
        ids, lab = (p + y)[:256], ([-100] * len(p) + y)[:256]
        if any(l != -100 for l in lab): feats.append({"input_ids": ids, "labels": lab})
    print(f"[train] {len(feats)} examples, mean len {np.mean([len(f['input_ids']) for f in feats]):.0f} tokens", flush=True)
    def collate(b):
        L = max(len(x["input_ids"]) for x in b)
        ids = torch.full((len(b), L), tok.pad_token_id); lab = torch.full((len(b), L), -100); att = torch.zeros((len(b), L), dtype=torch.long)
        for i, x in enumerate(b):
            n = len(x["input_ids"]); ids[i, :n] = torch.tensor(x["input_ids"]); lab[i, :n] = torch.tensor(x["labels"]); att[i, :n] = 1
        return {"input_ids": ids, "labels": lab, "attention_mask": att}
    model = prepare_model_for_kbit_training(base(), use_gradient_checkpointing=True)
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    model.print_trainable_parameters()
    class NanStop(TrainerCallback):
        def on_log(self, args, state, control, logs=None, **kw):
            if logs and "loss" in logs and (math.isnan(logs["loss"]) or logs["loss"] == 0.0):
                print("\n!!! LOSS IS NaN/0 -> stopping. Set MODEL = 'Qwen/Qwen2.5-1.5B-Instruct' and run again.\n", flush=True)
                open(f"{OUT}/NAN_{TAG}", "w").write("nan"); control.should_training_stop = True
    args = TrainingArguments(output_dir=f"{OUT}/ckpt_{TAG}", per_device_train_batch_size=4, gradient_accumulation_steps=4,
            num_train_epochs=1, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=30, logging_steps=20,
            save_steps=100, save_total_limit=2, fp16=True, max_grad_norm=0.3, optim="paged_adamw_8bit", report_to="none",
            seed=42, remove_unused_columns=False, gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False}, dataloader_num_workers=2, disable_tqdm=True)
    tr_ = Trainer(model=model, args=args, train_dataset=feats, data_collator=collate, callbacks=[NanStop()])
    ck = sorted(glob.glob(f"{OUT}/ckpt_{TAG}/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
    t0 = time.time(); tr_.train(resume_from_checkpoint=ck[-1] if ck else None)
    if os.path.exists(f"{OUT}/NAN_{TAG}"): sys.exit(3)
    model.save_pretrained(ADAPTER); print(f"[train] done in {(time.time()-t0)/60:.1f} min -> {ADAPTER}", flush=True)
    json.dump(tr_.state.log_history, open(f"{OUT}/train_log_{TAG}.json", "w"))
    del model, tr_; torch.cuda.empty_cache()

# ---------------- translation ----------------
model = PeftModel.from_pretrained(base(), ADAPTER).eval()
tok.padding_side = "left"
def translate(srcs, bs=24):
    order = np.argsort([-len(s) for s in srcs]); out = [None] * len(srcs); t0 = time.time()
    for k in range(0, len(srcs), bs):
        idx = order[k:k + bs]
        enc = tok([prompt(srcs[i]) for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            g = model.generate(**enc, max_new_tokens=180, do_sample=False, pad_token_id=tok.pad_token_id)
        for i, t in zip(idx, tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)):
            out[i] = t.strip().split("\n")[0].replace("Telugu:", "").strip()
        if (k // bs) % 10 == 0: print(f"   {k+len(idx)}/{len(srcs)}  ({time.time()-t0:.0f}s)", flush=True)
    return out

def load(run, sp):
    p = f"{PRED}/{run}__{sp}.csv"
    return pd.read_csv(p, dtype=str, keep_default_na=False) if os.path.exists(p) else None
SPLITS = [s for s in ["test", "extfull_FLORES200_devtest", "extfull_IN22_Gen", "extfull_IN22_Conv"] if load("baseline_s42", s) is not None]
SUB = {}
for sp in SPLITS:
    d = load("baseline_s42", sp)
    SUB[sp] = sorted(random.Random(2026).sample(range(len(d)), N_TS)) if (sp == "test" and len(d) > N_TS) else list(range(len(d)))
json.dump(SUB["test"], open(f"{OUT}/test_sample_indices.json", "w"))

for sp in SPLITS:
    fp = f"{OUT}/preds__{TAG}__{sp}.csv"
    if os.path.exists(fp): continue
    d = load("baseline_s42", sp).iloc[SUB[sp]]
    print(f"[gen] {TAG} {sp} n={len(d)}", flush=True)
    hyp = translate(d["en"].tolist())
    pd.DataFrame({"en": d["en"].tolist(), "te": d["te"].tolist(), "prediction": hyp}).to_csv(fp, index=False)
del model; torch.cuda.empty_cache()

# ---------------- scoring ----------------
from comet import download_model, load_from_checkpoint
comet = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da", saving_directory="/tmp/comet_models"))
def cm(src, hyp, ref):
    r = comet.predict([{"src": a, "mt": h, "ref": b} for a, h, b in zip(src, hyp, ref)], batch_size=32, gpus=1, progress_bar=False)
    return 100 * float(np.mean(r.scores))
rows = []
for sp in SPLITS:
    systems = [(f"{TAG} QLoRA ({N_TRAIN} pairs, 1 epoch)", pd.read_csv(f"{OUT}/preds__{TAG}__{sp}.csv", dtype=str, keep_default_na=False))]
    for run, lab in [("baseline_s42", "mBART-50 fine-tuned (seed 42)"), ("pscab_s42", "PS-CAB (seed 42)")]:
        x = load(run, sp)
        if x is not None: systems.append((lab, x.iloc[SUB[sp]]))
    for lab, d in systems:
        s, r, h = d["en"].tolist(), d["te"].tolist(), d["prediction"].tolist()
        empty = 100 * np.mean([len(x.strip()) == 0 for x in h])
        rows.append(dict(system=lab, split=sp, n=len(d), BLEU=round(sacrebleu.corpus_bleu(h, [r]).score, 2),
                         chrFpp=round(sacrebleu.corpus_chrf(h, [r], word_order=2).score, 2), COMET=round(cm(s, h, r), 2),
                         empty_output_pct=round(empty, 1)))
        print("[score]", rows[-1], flush=True)
T = pd.DataFrame(rows); T.to_csv(f"{OUT}/T8_llm_qlora_baseline.csv", index=False)
print("\n================ RESULTS (paste to Claude) ================\n### T8_llm_qlora_baseline\n" + T.to_string(index=False))
print("\nDone. Click Save Version.")
'''
open(f"{TMP}/worker.py", "w").write(WORKER)
json.dump({"model": MODEL, "pred": PRED, "train": TRAIN, "out": OUT, "n_train": N_TRAIN, "n_test": N_TEST_SAMPLE},
          open(f"{TMP}/cfg.json", "w"))
env = dict(os.environ, PYTHONUNBUFFERED="1", HF_HOME="/tmp/hf_cache", CUDA_VISIBLE_DEVICES="0",
           HF_TOKEN=HF_TOKEN, TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD="1")
print("\n[run] worker starting ...\n")
proc = subprocess.Popen([sys.executable, f"{TMP}/worker.py", f"{TMP}/cfg.json"], stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, env=env)
for line in proc.stdout: print(line, end="")
proc.wait()
if proc.returncode == 3: print("\n>>> Loss was NaN. Change MODEL at the top to 'Qwen/Qwen2.5-1.5B-Instruct' and run again.")
elif proc.returncode != 0: raise RuntimeError(f"worker exited with code {proc.returncode}; see output above")

[input] preds: /kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/pscab_q1/preds
[input] train: /kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/pscab_q1/data/train.csv
[info] no HF_TOKEN secret attached (not needed for unsloth/gemma-2-2b-it or Qwen).
[pip] installing (2-4 min) ...

[run] worker starting ...

I0000 00:00:1795015483.475195      85 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[ver] torch 2.11.0+cu130 | gpu Tesla T4
[train] 8000 examples, mean len 105 tokens
trainable params: 20,766,720 || all params: 2,635,108,608 || trainable%: 0.7881
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
{'loss': 1.242, 'grad_norm': 1.8209418058395386, 'learning_rate': 0.00013333333333333334, 'epoch': 0.04}
{'loss': 1.0149, 'grad_norm': 1.8384262323379517, 'learning_rate': 0.00019977668786231534, 'epoch': 0.08}
{'loss': 1.0106, 'grad_norm': 1.775103211402893, 'learning_rate': 0.00019799